In [1]:
import pandas as pd

# Define file path
file_path = "GitHub Master 250205Kai.xlsm"

# Load the Excel file and check available sheet names
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

# Display sheet names
sheet_names

['Definitions',
 'Visualisation Dashboard',
 'Financial Instruments',
 'High Level Dashboard',
 'Financing Baseline',
 'Funding Baseline',
 'Investment Needs',
 'FFRM (Input)',
 'OSeMOSYS (Input)']

In [5]:
from MinFin.minfin import load_excel_data
    
df_param_constraints, df_financing_baseline, df_funding_baseline, df_scenarios, df_currencies, df_technologies = load_excel_data(file_path)
df_financing_baseline

,Name,Description
0,Name of Project,
1,Name of Financier,
2,Technology,
3,Source,
4,Financing Source,
5,Type of Finance,
6,Financing Sector,
7,Financial Institution,
8,Origin of Finance,
9,Volume of Finance,


In [3]:
df_financing_baseline_full = pd.read_excel(file_path, sheet_name="Financing Baseline")
df_financing_baseline_full

,MINFin,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 139,Unnamed: 140,Unnamed: 141,Unnamed: 142,Unnamed: 143,Unnamed: 144,Unnamed: 145,Unnamed: 146,Unnamed: 147,Unnamed: 148
0,Financing Baseline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FORWARD LOOKING PARAMETERS FOR FINANCING SOURCES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Repayment Currency,Foreign Currency,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Exchange Rate Projections,USD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
511,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
513,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [141]:
import time
import random
import numpy as np
# Generate exchange-rate data for 2014 through 2030
currency_list = ["KES", "USD", "EUR", "GBP", "JPY"]
base_rates = {
    "KES": 2,#0.009,
    "USD": 3.0,
    "EUR": 1.08,
    "GBP": 1.36,
    "JPY": 0.007
}
exchange_rates_by_year = pd.DataFrame(columns=currency_list)
for year in range(2010, 2080):
    exchange_rates_by_year[year] = {
        currency: round(base_rates[currency] * round(1 + random.uniform(-0.05, 0.05)), 5) for currency in currency_list
    }
    
def get_exchange_rates(target_currency, currency_series,year_series):
    """
    Look up the exchange rate for each year and source currency and convert to the target currency.
    
    Parameters:
    year_series: Pandas Series of years
    currency_series: Pandas Series of source currencies
    target_currency: Target currency (e.g. USD)
    
    Returns:
    Pandas Series of exchange rates
    """
    # print(year_series)
    year_series = year_series.astype(int)  # Ensure years are integers
    mask = year_series.isin(exchange_rates_by_year.keys()) & currency_series.isin(currency_list)
    # print(mask)
    rates = year_series[mask].map(lambda y: exchange_rates_by_year[y]).combine(currency_series[mask], lambda rates_dict, c: rates_dict.get(c, None))
    target_rates = year_series[mask].map(lambda y: exchange_rates_by_year[y].get(target_currency, 1))
    return  target_rates/rates


class financing_baseline_extractor:
    def __init__(self,df_financing_baseline_full,currency='KES',starting_year=2024,number_of_payments_per_annum=1) -> None:
        self.currency = currency
        self.starting_year = starting_year
        self.foreign_currency = 'USD'
        self.discount_rate = 5.33/100 #'High Level Dashboard'!E34
        self.number_of_payments_per_annum = number_of_payments_per_annum
        self.starting_rows = {'exchange_rate': 36,'historical baseline': 46}
        self.starting_cols = {'historical baseline':0}
        # {'least_cost': {'variable_cost': 4, 'fixed_cost': 6, 'annual_elec_production': 10,'co2_emission':18}}
        # self.starting_cols['net_zero']= { key:value-1 for key,value in self.starting_cols['least_cost'].items()}
        # self.starting_cols[scenario]['carbon_price'] = 16
        self.df_financing_baseline_full = df_financing_baseline_full
        self.historical = self.get_historical()
        
        
        
    def get_historical(self):
        df_financing_baseline_full=self.df_financing_baseline_full
        starting_row = self.starting_rows['historical baseline']
        starting_col = self.starting_cols['historical baseline']
        # Use the first row as column names
        new_columns = df_financing_baseline_full.iloc[starting_row, starting_col:starting_col+20].values
        
        # Extract data (skip the header row)
        df_historical = df_financing_baseline_full.iloc[starting_row+1:starting_row+400, starting_col:starting_col+20].copy()
        
        # Reassign column names
        df_historical.columns = [x.strip() for x in new_columns]        
        df_historical = df_historical.drop(columns=["Volume in KES", "Volume in USD", "Exchange Rate","Maturity"], errors='ignore').dropna(how='all', axis=0)#.dropna(how='all', axis=1)
        df_historical = self.add_columns_for_other_currencies(df_historical, exchange_rates_by_year)
        return df_historical.dropna(how='all', axis=0).reset_index(drop=True)#.dropna(how='all', axis=1)
    
    def add_columns_for_other_currencies(self, df, exchange_rates_by_year):
        """
        Compute financial fields and add:
        - volume of finance in currency
        - volume of finance in foreign currency
        - exchange rate to currency
        - exchange rate to foreign currency
        
        Parameters:
        df: DataFrame of transaction data
        exchange_rates: Currency exchange-rate DataFrame (base_currency to other currencies)
        base_currency: Base currency, default KES
        foreign_currency: Foreign currency, usually USD
        """
        base_currency=self.currency
        foreign_currency=self.foreign_currency

        # 1. Exchange rate of the instrument currency vs base_currency
        # df[f"exchange rate to {base_currency}"] = df["Currency"].apply(lambda row: get_exchange_rate(row, "Currency", "Year"))

        # # 2. Exchange rate vs foreign_currency
        # df[f"Exchange rate to {foreign_currency}"] = df[f"exchange rate to {base_currency}"].apply(lambda row: get_exchange_rate(row, foreign_currency, "Year"))

        # 3. Compute volume of finance in local currency
        df[f"Volume in {base_currency}"] = df["Volume of Finance"] *get_exchange_rates(base_currency,df["Currency"],df["Year"])

        # 4. Compute volume of finance in foreign currency (usually USD)
        df[f"Volume in {foreign_currency}"] = df["Volume of Finance"] *get_exchange_rates(foreign_currency,df["Currency"],df["Year"])
        df[f"Exchange Rate to {base_currency}"] = get_exchange_rates(base_currency,df["Currency"],df["Year"])
        df[f"Exchange Rate to {foreign_currency}"] = get_exchange_rates(foreign_currency,df["Currency"],df["Year"])
        df["Maturity"] = df["Term"]-self.starting_year+df["Year"]
        
        return df
    
    def cal_repayment_schedule(self,df):
        df=df.reset_index(drop=True)
        years =  list(range(2010, 2071))
        df_repayment = pd.DataFrame(columns=years)
        df_repayment['Repayment'] = 0
        df_repayment['Name of Project'] = df['Name of Project']
        
        repay_years_list = []
        # Iterate over each row in the DataFrame
        for index, row in df.fillna(0).iterrows():
            # Calculate the repayment years for each project
            repay_years = list(range(row['Year'], row['Year'] + int(row['Term']+1)))
            repay_years_list.append(repay_years)
        
        # Assign the list of repayment years to the DataFrame
        df_repayment['repay_years'] = repay_years_list
        df_repayment['Project ID'] = df.index
        for year in range(2010, 2071):
            # print(df['Volume in KES'])
            for project_id in df.index:
                # print( 2020 in list(df_repayment.loc[df_repayment['Project ID'] == project_id,'repay_years'])[0])
                # print('==',list(df_repayment.loc[df_repayment['Project ID'] == project_id,'repay_years'])[0])
                repay_years = list(df_repayment.loc[df_repayment['Project ID'] == project_id,'repay_years'])[0]
                if year in repay_years:
                    if year == repay_years[-1]:
                        df_repayment.loc[df_repayment['Project ID'] == project_id,year] = self.cal_repayment_value(df.loc[project_id,"Rate"],df['Volume of Finance'],df.loc[project_id,"Schedule"],year,repay_years,term=df.loc[project_id,'Term'],grace_period=df.loc[project_id,'Grace period']) 
                    else:
                        df_repayment.loc[df_repayment['Project ID'] == project_id,year] = self.cal_repayment_value(df.loc[project_id,"Rate"],df['Volume of Finance'],df.loc[project_id,"Schedule"],year,repay_years,term=df.loc[project_id,'Term'],grace_period=df.loc[project_id,'Grace period']) 
                else:
                    df_repayment.loc[df_repayment['Project ID'] == project_id,year] = 0
            # df_repayment.loc[df_repayment[year] == year] = df['Volume in KES'] * (1 + df['Rate']) ** df['Maturity']
            
        df_repayment['Sum of Repayment'] = df_repayment[years].sum(axis=1).astype(float)
        # print("============================================")
        df_repayment['Market Element'] = self.cal_market_element()
        df_repayment['Grant Element'] = self.cal_grant_element()
        # print('term',df['Term'].astype(float),df['Term'].astype(float).replace(0,100000))
        df_repayment['Average Annual Payment']= df_repayment['Sum of Repayment'] /df['Term'].astype(float)#.replace(0,100000)
        df_repayment['Average Annual Payment'] = df_repayment['Average Annual Payment'].replace([np.inf, -np.inf], np.nan).fillna(0)
        return df_repayment.reset_index(drop=True)
    def cal_repayment_value(self,interest_rate,volume,scenario,year,repay_years,term=0,grace_period=0):
        # print(type(scenario),scenario)
        # print(type(interest_rate),interest_rate)
        if scenario in ['Equity']:

             return volume * interest_rate 
        elif scenario in ['Lump Sum Principal']:
            if year == repay_years[-1]:
                return volume * (1+interest_rate) 
            else:
                return volume * interest_rate 
        
        elif scenario in ['Lump Sum Principal and Interest']:
            if year == repay_years[-1]:
                return volume * (1+interest_rate)**term
            else:
                return 0
                    
        elif scenario in ['EPP with Grace Years for Principal']:
            if year <= repay_years[0]+grace_period:
                return volume * interest_rate
            else:
                return -self.calculate_annuity_payment(interest_rate, term-grace_period+1, volume, fv=0)
        
        elif scenario in ['Equal Principal Payments (EPP)']:    
                        
            return -self.calculate_annuity_payment(interest_rate, term-grace_period+1, volume, fv=0)
        elif scenario in ['EPP with Grace Years for Principal and on Interest']:
            if year <= repay_years[0]+grace_period:
                return volume * interest_rate
            else:
                return -self.calculate_annuity_payment(interest_rate, term-grace_period+1, volume, fv=0)
        else:
            return None
            
    def get_discount_rate_for_grant_ele(self):
        # discount_rate = historical['Rate'].max(historical.loc[:,'Type of Finance']=='Loan')
        # Filter rows where Type of Finance is Loan
        loan_data = self.historical[self.historical['Type of Finance'] == 'Loan']
        # Check whether any Loan financing exists
        if not loan_data.empty:
            # If present, take the maximum Rate
            discount_rate = loan_data['Rate'].max()
        else:
            # Otherwise use default 10%
            discount_rate = 0.10
        
        return discount_rate
    
    def get_d(self,number_of_payments=None):
        if number_of_payments is None:
            # number_of_payments = self.arguments.get('number_of_payments_per_annum', 1)
            number_of_payments = self.number_of_payments_per_annum
        return (1+self.get_discount_rate_for_grant_ele())**(1/number_of_payments)-1
    
    def cal_grant_element(self,number_of_payments=None):
        if number_of_payments is None:
            # number_of_payments = self.arguments.get('number_of_payments_per_annum', 1)
            number_of_payments = self.number_of_payments_per_annum
        d = self.get_d(number_of_payments)
        
        interest_rate = self.historical['Rate'].dropna().reset_index(drop=True)
        grace_period = self.historical['Grace period'].dropna().reset_index(drop=True) 
        loan_term = self.historical['Term'].dropna().reset_index(drop=True)
        financing_schedule_type = self.historical['Schedule'].dropna()
        
        
        term1 = 1 - interest_rate / (number_of_payments*d)
        
        factor1 = 1 / ((1 + d) ** (number_of_payments * grace_period))
        factor2 = 1 / ((1 + d) ** (number_of_payments * (loan_term + grace_period)))
        denominator = d * number_of_payments * loan_term #+ grace_period) - number_of_payments * grace_period)
        denominator = denominator.where(denominator != 0, 1)
        # print(interest_rate,grace_period,loan_term)
        term2 = 1 - ((factor1 - factor2) / denominator)

        epp_grant_element = pd.DataFrame(term1 * term2)
        lump_grant_element = pd.DataFrame(1-(1+interest_rate*(loan_term + grace_period))/(1+self.get_discount_rate_for_grant_ele())**(loan_term + grace_period))
        equity = 'equity'
        grant_element = pd.DataFrame(columns=['Grant Element'])
        
        factor = pd.DataFrame([1 if "EPP" in t else 0 for t in financing_schedule_type])
        print(factor)
        grant_element['Grant Element'] = epp_grant_element*factor + lump_grant_element*(1-factor)

        grant_element['Grant Element'] = np.where(financing_schedule_type.str.contains("Equity"), "Equity", grant_element['Grant Element'])
        return  grant_element
    def cal_market_element(self,number_of_payments=None):
        return pd.DataFrame([1 - item if isinstance(item, (int, float)) else item for item in self.cal_grant_element(number_of_payments)['Grant Element']])
    
    def cal_discounted_schedule(self,df_repayment,discount_rate,current_year=2024):
        df = self.historical.reset_index(drop=True)
        # years = [package for package in df_repayment.columns if package.isdigit()]
        df_discounted = df_repayment.copy()
        df_discounted.drop(columns=['Repayment','Name of Project','repay_years','Project ID','Sum of Repayment','Average Annual Payment'],inplace=True)
        df_discounted = df_discounted / (1 + discount_rate) ** ((df_discounted.columns - current_year+1) * (df_discounted.columns-current_year>=0))
        df_discounted['Sum of Repayment'] = df_discounted.sum(axis=1).astype(float)
        df_discounted['Average Annual Payment']= df_discounted['Sum of Repayment'] /df['Term'].astype(float)
        return df_discounted
    
    @staticmethod
    def calculate_annuity_payment(rate, nper, pv, fv=0):
        """
        Annuity Payment
        rate: interest rate for each period
        nper: total number of payment periods
        pv: present value (default 0)
        fv: future value (default 0)
        """
        if rate == 0:
            return -(pv + fv) / nper
        else:
            return -(rate * (pv * (1 + rate) ** nper + fv)) / ((1 + rate) ** nper - 1)
    @staticmethod
    def select_grant_element(index,schedule_type,epp_grant_element,lump_grant_element):
        
        if "Equity" in schedule_type:
            print('Equity')
            return "Equity"        
        
    # def cal_general_repayment_statistics(self):
    #     self.weighte_averages = self.cal_weighted_average()
    #     return self.weighte_averages
    

class financing_baseline_stats:
    def __init__(self, financing_baseline_extractor, repayment_schedule):
        self.repayment_schedule = repayment_schedule.copy()
        self.historical = financing_baseline_extractor.get_historical()        
        # Fill multiple columns in one pass
        cols_to_copy = ['Financing Source', 'Type of Finance', 'Volume in USD', 'Term', 'Grace period']
        self.repayment_schedule.loc[:, cols_to_copy] = self.historical[cols_to_copy]

                
        self.repayment_schedule['Interest rate'] = self.historical['Rate']
        # self.repayment_schedule['Financing Source'] = self.historical['Financing Source']
        # self.repayment_schedule['Type of Finance'] = self.historical['Type of Finance']
        # self.repayment_schedule['Volume in USD'] = self.historical['Volume in USD']
        # self.repayment_schedule['Term'] = self.historical['Term']
        # self.repayment_schedule['Grace period'] = self.historical['Grace period']
        self.cols = ['Volume (USD)', 'Interest rate', 'Term', 'Grace period', 'Average Annual Payment']#, 'Average Annual Discounted Payment',	'Total Discounted Payment']
        self.rows = ['Total financing volumes',
                'Conc_IFI',
                'Conc_DPS',
                'Comm_Intl',
                'Comm_Dom',
                ]
    def cal_summary_stats(self):
        cols = self.cols
        rows = self.rows
        repayment_schedule = self.repayment_schedule
        
        df_summary_stats = pd.DataFrame(columns=cols,index=rows)
        for row in rows:
            # Calculate 'Volume (USD)' for 'Commercial Domestic Finance (Comm Dom)'
            df_summary_stats.loc[row, 'Volume (USD)'] = self.historical[
                self.historical['Financing Source'] == row
            ]['Volume in USD'].sum()
        
        for row in rows[1:]:
            for col in cols[1:]:
                weighted_data = repayment_schedule[
                (repayment_schedule['Financing Source'] == row)
                ][col]* repayment_schedule[
                (repayment_schedule['Financing Source'] == row)  
                ][ "Volume in USD"]  
                df_summary_stats.loc[row, col] = weighted_data.sum() / repayment_schedule[
                (repayment_schedule['Financing Source'] == row) 
                ][ "Volume in USD"].sum() 
        
            debt_share = repayment_schedule[
                    (repayment_schedule['Financing Source'] == row)
                    ][repayment_schedule['Type of Finance'] == 'Loan']['Volume in USD'].sum() / repayment_schedule[
                    (repayment_schedule['Financing Source'] == row) 
                    ][ "Volume in USD"].sum()
            df_summary_stats.loc[row, 'Debt Share'] = debt_share
            df_summary_stats.loc[row, 'Equity Share'] = 1 - debt_share
        df_summary_stats.loc[rows[0], 'Volume (USD)'] = df_summary_stats['Volume (USD)'].sum()    
        
        return df_summary_stats
    
    def cal_equity_debt_stats(self,type_of_finance='Equity'):
        cols = self.cols
        rows = self.rows
        repayment_schedule = self.repayment_schedule
        df = pd.DataFrame(columns=cols,index=rows)
        for row in rows[1:]:
            # Calculate 'Volume (USD)' for 'Commercial Domestic Finance (Comm Dom)'
            df.loc[row, 'Volume (USD)'] = self.historical[
                (self.historical['Financing Source'] == row) & (self.historical['Type of Finance'] == type_of_finance)
            ]['Volume in USD'].sum()
            
        df.loc[rows[0], 'Volume (USD)'] = df['Volume (USD)'].sum()
        # repayment_schedule['Interest rate'] = self.historical['Rate']
        # repayment_schedule['Financing Source'] = self.historical['Financing Source']
        # repayment_schedule['Type of Finance'] = self.historical['Type of Finance']
        # repayment_schedule['Volume in USD'] = self.historical['Volume in USD']
        # repayment_schedule['Term'] = self.historical['Term']
        # repayment_schedule['Grace period'] = self.historical['Grace period']
        for row in rows[1:]:
            for col in cols[1:]:
                
                weighted_data = repayment_schedule[
                (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)
                ][col]* repayment_schedule[
                (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)
                ][ "Volume in USD"]  
                
                df.loc[row, col] = weighted_data.sum() / repayment_schedule[
                (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)
                ][ "Volume in USD"].sum() 
                
                 
                if df.loc[row, col] == float('inf'):
                    print("sss",repayment_schedule[
                (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)
                ][ "Volume in USD"].sum(),weighted_data.sum())  
                    print(repayment_schedule[
                (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)
                ][col])
                    print(repayment_schedule[
                (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)
                ][ "Volume in USD"])
                    
                # Select the correct volume column based on currency condition
                volume_column = "Volume in USD"
                
            if type_of_finance in ["Loan", "loan"]:
                market_element  = repayment_schedule[
            (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)& (historical['Schedule'] != 'Equity')
            ]['Market Element']* repayment_schedule[
            (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)& (historical['Schedule'] != 'Equity')
            ][ "Volume in USD"]/ repayment_schedule[
                (repayment_schedule['Financing Source'] == row) & (repayment_schedule['Type of Finance'] == type_of_finance)
                ][ "Volume in USD"].sum() 
            
                df.loc[row, 'Market Element'] = market_element.sum()
                df.loc[row, 'Grant Element'] = 1 - df.loc[row, 'Market Element']
                # Compute weighted sum
                # numerator = (filtered_df[volume_column] * repayment_schedule[col]).sum()
                
                # Compute total volume sum
                # denominator = filtered_df[volume_column].sum()
                
                # Avoid division by zero
                # df.loc[row, col] = numerator / denominator if denominator != 0 else 0    
        return df
    def add_weighted_average(self,df):
        cols = df.columns
        rows = self.rows
        for index, col in enumerate(cols[1:]):
            print(index,col)
            df.iloc[0, index+1] = (df[col][1:]*df['Volume (USD)'][1:]).sum()/df['Volume (USD)'][1:].sum()
            #  print(df.iloc[1, index+1])
        return df
    def get_repayment_statistics(self):
        print(self.cal_summary_stats().columns)
        df_summary = self.add_weighted_average(self.cal_summary_stats())
        df_equity = self.add_weighted_average(self.cal_equity_debt_stats())
        df_debt = self.add_weighted_average(self.cal_equity_debt_stats(type_of_finance='Loan'))
        df_final = pd.concat(
        [df_summary, df_equity, df_debt], 
        axis=0, 
        keys=['Summary', 'Equity', 'Debt']  # Hierarchical index labels
        )
        return df_final       
        
    def get_institution_shares(self):
        rows = ["Bilateral Agency",
                "Multilateral Agency",
                "Foreign Government",
                "National Government",
                "Domestic Public Sector",
                "Climate Funds",
                "Commercial Bank",
                "Private Equity Fund"
                ]
        repayment_schedule = self.repayment_schedule
        df_institution_shares = pd.DataFrame(index=rows)

        for row in rows:   
            df_institution_shares.loc[row, 'Share'] = repayment_schedule[(self.historical['Financial Institution'] == row) 
                ][ "Volume in USD"].sum()/ repayment_schedule.loc[:, "Volume in USD"].sum() 
        
        return df_institution_shares
    
    def get_financing_sector_shares(self):
        rows = ["Public",
                "Private",
                "Comm_Dom",
                "Comm_Intl",
                "Conc_DPS",
                "Conc_IFI",
                ]
        repayment_schedule = self.repayment_schedule
        df_financing_sector_shares = pd.DataFrame(index=rows)
        print(self.historical.columns)
        for row in rows: 
              
            df_financing_sector_shares.loc[row,'Share'] = repayment_schedule[(self.historical['Financing Sector'] == row) | (self.historical['Financing Source'] == row)
            ][ "Volume in USD"].sum()/ repayment_schedule.loc[ :,"Volume in USD"].sum() 
        
        df_financing_sector_shares.loc["Domestic","Share"] = df_financing_sector_shares.loc["Comm_Dom", "Share"] + df_financing_sector_shares.loc["Conc_DPS", "Share"]
        df_financing_sector_shares.loc["International","Share"] = df_financing_sector_shares.loc["Comm_Intl", "Share"] + df_financing_sector_shares.loc["Conc_IFI", "Share"]
        df_financing_sector_shares.drop(["Comm_Dom", "Comm_Intl", "Conc_DPS", "Conc_IFI"], inplace=True)
        return df_financing_sector_shares        
        


    
fb = financing_baseline_extractor(df_financing_baseline_full)
historical = fb.get_historical()
# # historical
# repayment_schedule = fb.cal_repayment_schedule(historical)
# repayment_schedule.T.head(50)
fbs= financing_baseline_stats(fb,repayment_schedule)
fbs.get_repayment_statistics()
# historical


Index(['Volume (USD)', 'Interest rate', 'Term', 'Grace period',
       'Average Annual Payment', 'Debt Share', 'Equity Share'],
      dtype='object')
0 Interest rate
1 Term
2 Grace period
3 Average Annual Payment
4 Debt Share
5 Equity Share
0 Interest rate
1 Term
2 Grace period
3 Average Annual Payment
0 Interest rate
1 Term
2 Grace period
3 Average Annual Payment
4 Market Element
5 Grant Element


C:\Users\Zixuan\AppData\Local\Temp\ipykernel_45484\4120385727.py:317: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
C:\Users\Zixuan\AppData\Local\Temp\ipykernel_45484\4120385727.py:317: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
C:\Users\Zixuan\AppData\Local\Temp\ipykernel_45484\4120385727.py:317: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
C:\Users\Zixuan\AppData\Local\Temp\ipykernel_45484\4120385727.py:317: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
C:\Users\Zixuan\AppData\Local\Temp\ipykernel_45484\4120385727.py:317: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
C:\Users\Zixuan\AppData\Local\Temp\ipykernel_45484\4120385727.py:317: UserWarning: Boolean Seri

Volume (USD) Interest rate       Term  \
Summary Total financing volumes  10983.174661      0.053333  27.139904   
        Conc_IFI                  8169.432921      0.024346  27.211158   
        Conc_DPS                   1220.21174      0.168093   32.11396   
        Comm_Intl                    1337.892      0.108661  22.029274   
        Comm_Dom                      255.638      0.142344  27.867336   
Equity  Total financing volumes   2207.860073      0.157236  29.849349   
        Conc_IFI                   135.518333       0.15093    24.8871   
        Conc_DPS                   1014.23174      0.196846  34.037672   
        Comm_Intl                     853.872      0.115266  25.574665   
        Comm_Dom                      204.238      0.140188  30.214505   
Debt    Total financing volumes   8775.314587      0.027191   26.45821   
        Conc_IFI                  8033.914587      0.022211  27.250361   
        Conc_DPS                       205.98      0.026511  22.641732   
        Comm_Intl                      484.02      0.097009  15.774761   
        Comm_Dom                         51.4      0.150913  18.540856   

                                Grace period Average Annual Payment  \
Summary Total financing volumes     5.355647               9.899434   
        Conc_IFI                    6.807053               8.663707   
        Conc_DPS                    1.198324              21.692584   
        Comm_Intl                   1.019163               7.075876   
        Comm_Dom                    1.511904               7.875709   
Equity  Total financing volumes          0.0              15.083797   
        Conc_IFI                         0.0               4.483015   
        Conc_DPS                         0.0              25.283848   
        Comm_Intl                        0.0               6.290694   
        Comm_Dom                         0.0               8.226935   
Debt    Total financing volumes     6.703122               8.595053   
        Conc_IFI                    6.921876               8.734228   
        Conc_DPS                    7.098791               4.009441   
        Comm_Intl                   2.817094               8.461036   
        Comm_Dom                    7.519455               6.480112   

                                 Debt Share  Equity Share  Market Element  \
Summary Total financing volumes    0.798978      0.201022             NaN   
        Conc_IFI                   0.983412      0.016588             NaN   
        Conc_DPS                   0.168807      0.831193             NaN   
        Comm_Intl                  0.361778      0.638222             NaN   
        Comm_Dom                   0.201066      0.798934             NaN   
Equity  Total financing volumes         NaN           NaN             NaN   
        Conc_IFI                        NaN           NaN             NaN   
        Conc_DPS                        NaN           NaN             NaN   
        Comm_Intl                       NaN           NaN             NaN   
        Comm_Dom                        NaN           NaN             NaN   
Debt    Total financing volumes         NaN           NaN        0.223720   
        Conc_IFI                        NaN           NaN        0.193863   
        Conc_DPS                        NaN           NaN        0.228687   
        Comm_Intl                       NaN           NaN        0.661032   
        Comm_Dom                        NaN           NaN        0.752387   

                                 Grant Element  
Summary Total financing volumes            NaN  
        Conc_IFI                           NaN  
        Conc_DPS                           NaN  
        Comm_Intl                          NaN  
        Comm_Dom                           NaN  
Equity  Total financing volumes            NaN  
        Conc_IFI                           NaN  
        Conc_DPS                           NaN  
        Comm_Intl                    

In [98]:
fbs.get_institution_shares()

,Share
Bilateral Agency,0.405901
Multilateral Agency,0.317207
Foreign Government,0.016177
National Government,0.100719
Domestic Public Sector,0.000000
Climate Funds,0.023221
Commercial Bank,0.031322
Private Equity Fund,0.105453


In [139]:
fbs.get_financing_sector_shares()

Index(['Name of Project', 'Name of Financier', 'Technology', 'Source', 'Year',
       'Financing Source', 'Type of Finance', 'Financing Sector',
       'Financial Institution', 'Origin of Finance', 'Volume of Finance',
       'Currency', 'Rate', 'Term', 'Grace period', 'Schedule', 'Volume in KES',
       'Volume in USD', 'Exchange Rate to KES', 'Exchange Rate to USD',
       'Maturity'],
      dtype='object')


,Share
Public,0.863024
Private,0.136976
Domestic,0.134374
International,0.865626


In [122]:
print(repayment_schedule.iloc[108,:])

2010                         0
2011                         0
2012                         0
2013                         0
2014                         0
                          ... 
Project ID                 108
Sum of Repayment          11.3
Market Element             1.0
Grant Element              0.0
Average Annual Payment     0.0
Name: 108, Length: 69, dtype: object


In [25]:
# example.head(50)
historical.head(30)
fb.cal_discounted_schedule(repayment_schedule,0.0533,current_year=2024).head(30)
fb.cal_grant_element(1).head(20)
fb.cal_market_element(1).head(20)

     0
0    1
1    0
2    1
3    0
4    0
..  ..
177  0
178  1
179  1
180  1
181  0

[182 rows x 1 columns]
     0
0    1
1    0
2    1
3    0
4    0
..  ..
177  0
178  1
179  1
180  1
181  0

[182 rows x 1 columns]


,0
0,0.947912
1,Equity
2,0.339693
3,Equity
4,Equity
5,0.179208
6,0.483751
7,0.592909
8,Equity
9,0.592909


In [24]:
historical.head(30)
financing_baseline_extractor.cal_discounted_schedule(repayment_schedule,0.0533,current_year=2024).T.head(30)
fb.cal_grant_element(1).head(20)
fb.cal_market_element(1).head(20)

TypeError: financing_baseline_extractor.cal_discounted_schedule() missing 1 required positional argument: 'discount_rate'